# Работа с признаками. Практика

In [1]:
import os

import pandas as pd
import psycopg
import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder, 
    SplineTransformer, 
    QuantileTransformer, 
    RobustScaler,
    PolynomialFeatures,
    KBinsDiscretizer,
)

from dotenv import load_dotenv

load_dotenv()

TABLE_NAME = "users_churn"

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

EXPERIMENT_NAME = "Feature_engineering_lession_5"
RUN_NAME = "preprocessing"
REGISTRY_MODEL_NAME = "preprocessing_feature_engineering_lession_5_model"

ASSETS_DIR = "assets"

os.makedirs(ASSETS_DIR, exist_ok=True)

pd.options.display.max_columns = 100
pd.options.display.max_rows = 64

/home/mle-user/projects/mle-mlflow/.venv/lib/python3.10/site-packages/pydantic/_internal/_config.py:408: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


In [2]:
connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
    "sslmode": "disable"
}

connection.update(postgres_credentials)

with psycopg.connect(**connection) as conn:

    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM destination.{TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)

df.head(2)

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,4,5575-GNVDE,2017-04-01,NaT,One year,No,Mailed check,56.95,1889.50,DSL,Yes,No,Yes,No,No,No,Male,0,No,No,No,0
1,5,3668-QPYBK,2019-11-01,2020-01-01,Month-to-month,Yes,Mailed check,53.85,108.15,DSL,Yes,Yes,No,No,No,No,Male,0,No,No,No,1


In [3]:
# определение категориальных колонок, которые будут преобразованы
cat_columns = ["type", "payment_method", "internet_service", "gender"]

# создание объекта OneHotEncoder для преобразования категориальных переменных
# auto - автоматическое определение категорий
# ignore - игнорировать ошибки, если встречается неизвестная категория
# max_categories - максимальное количество уникальных категорий
# sparse_output - вывод в виде разреженной матрицы, если False, то в виде обычного массива
# drop="first" - удаляет первую категорию, чтобы избежать ловушки мультиколлинеарности
encoder_oh = OneHotEncoder(
    categories="auto",
    handle_unknown="ignore",
    max_categories=10,
    sparse_output=False,
    drop="first"
)

# применение OneHotEncoder к данным. Преобразование категориальных данных в массив
encoded_features = encoder_oh.fit_transform(df[cat_columns].to_numpy())

# преобразование полученных признаков в DataFrame и установка названий колонок
# get_feature_names_out() - получение имён признаков после преобразования
encoded_df = pd.DataFrame(
    encoded_features,
    columns=encoder_oh.get_feature_names_out(cat_columns),
    index=df.index
)

# конкатенация исходного DataFrame с новым DataFrame, содержащим закодированные категориальные признаки
# axis=1 означает конкатенацию по колонкам
obj_df = pd.concat([df, encoded_df], axis=1)

obj_df.head(2)

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target,type_One year,type_Two year,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,internet_service_Fiber optic,internet_service_None,gender_Male
0,4,5575-GNVDE,2017-04-01,NaT,One year,No,Mailed check,56.95,1889.50,DSL,Yes,No,Yes,No,No,No,Male,0,No,No,No,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,5,3668-QPYBK,2019-11-01,2020-01-01,Month-to-month,Yes,Mailed check,53.85,108.15,DSL,Yes,Yes,No,No,No,No,Male,0,No,No,No,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


In [4]:
num_columns = ["monthly_charges", "total_charges"]

n_knots = 3
degree_spline = 4
n_quantiles = 100
degree = 3
n_bins = 5
encode = 'ordinal'
strategy = 'uniform'
subsample = None

# 0. Копируем obj_df в num_df и заполняем пропуски (NaN) в числовых колонках
num_df = obj_df.copy()
num_df[num_columns] = num_df[num_columns].fillna(0)  # или .median(), если по условию требуется медиана


# SplineTransformer
encoder_spl = SplineTransformer(n_knots=n_knots, degree=degree_spline)
encoded_features = encoder_spl.fit_transform(num_df[num_columns].to_numpy())

encoded_df = pd.DataFrame(
    encoded_features, 
    columns=encoder_spl.get_feature_names_out(num_columns), 
    index=num_df.index
)
num_df = pd.concat([num_df, encoded_df], axis=1)


# QuantileTransformer
encoder_q = QuantileTransformer(n_quantiles=n_quantiles)
encoded_features = encoder_q.fit_transform(num_df[num_columns].to_numpy())

encoded_df = pd.DataFrame(encoded_features, index=num_df.index)
encoded_df.columns = [col + f"_q_{n_quantiles}" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


# RobustScaler
encoder_rb = RobustScaler()
encoded_features = encoder_rb.fit_transform(num_df[num_columns].to_numpy())

encoded_df = pd.DataFrame(encoded_features, index=num_df.index)
encoded_df.columns = [col + f"_robust" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


# PolynomialFeatures
encoder_pol = PolynomialFeatures(degree=degree)
encoded_features = encoder_pol.fit_transform(num_df[num_columns].to_numpy())

encoded_df = pd.DataFrame(
    encoded_features, 
    columns=encoder_pol.get_feature_names_out(num_columns), 
    index=num_df.index
)
# Берем все полиномиальные признаки после константы (1) и исходных признаков
encoded_df = encoded_df.iloc[:, 1 + len(num_columns):]
encoded_df.columns = [col + "_poly" for col in encoded_df.columns]
num_df = pd.concat([num_df, encoded_df], axis=1)


# KBinsDiscretizer
encoder_kbd = KBinsDiscretizer(
    n_bins=n_bins, 
    encode=encode, 
    strategy=strategy, 
    subsample=subsample
)
encoded_features = encoder_kbd.fit_transform(num_df[num_columns].to_numpy())

encoded_df = pd.DataFrame(encoded_features, index=num_df.index)
encoded_df.columns = [col + f"_bin" for col in num_columns]
num_df = pd.concat([num_df, encoded_df], axis=1)

num_df.head(2)

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target,type_One year,type_Two year,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,internet_service_Fiber optic,internet_service_None,gender_Male,monthly_charges_sp_0,monthly_charges_sp_1,monthly_charges_sp_2,monthly_charges_sp_3,monthly_charges_sp_4,monthly_charges_sp_5,total_charges_sp_0,total_charges_sp_1,total_charges_sp_2,total_charges_sp_3,total_charges_sp_4,total_charges_sp_5,monthly_charges_q_100,total_charges_q_100,monthly_charges_robust,total_charges_robust,monthly_charges^2_poly,monthly_charges total_charges_poly,total_charges^2_poly,monthly_charges^3_poly,monthly_charges^2 total_charges_poly,monthly_charges total_charges^2_poly,total_charges^3_poly,monthly_charges_bin,total_charges_bin
0,4,5575-GNVDE,2017-04-01,NaT,One year,No,Mailed check,56.95,1889.50,DSL,Yes,No,Yes,No,No,No,Male,0,No,No,No,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.000116,0.094742,0.554677,0.335807,0.014658,0.0,0.004242,0.228653,0.596333,0.169278,1.493678e-03,0.0,0.394114,0.581845,-0.246550,0.146087,3243.3025,107607.0250,3.570210e+06,184706.077375,6.128220e+06,2.033235e+08,6.745912e+09,1.0,1.0
1,5,3668-QPYBK,2019-11-01,2020-01-01,Month-to-month,Yes,Mailed check,53.85,108.15,DSL,Yes,Yes,No,No,No,No,Male,0,No,No,No,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.000301,0.114432,0.572271,0.302499,0.010496,0.0,0.037668,0.445733,0.470623,0.045975,1.603155e-08,0.0,0.355221,0.120864,-0.303588,-0.379687,2899.8225,5823.8775,1.169642e+04,156155.441625,3.136158e+05,6.298524e+05,1.264968e+06,1.0,0.0


In [5]:
# Заполняем пропуски в исходном датафрейме перед запуском препроцессора
df[num_columns] = df[num_columns].fillna(0)

# 1. Объединяем преобразования над числовыми колонками в ColumnTransformer
numeric_transformer = ColumnTransformer(
    transformers=[
        ("spl", encoder_spl, num_columns),
        ("q", encoder_q, num_columns),
        ("rb", encoder_rb, num_columns),
        ("pol", encoder_pol, num_columns),
        ("kbd", encoder_kbd, num_columns),
    ]
)

# 2. Оборачиваем преобразование категориальных колонок в Pipeline
categorical_transformer = Pipeline(steps=[('encoder', encoder_oh)])

# 3. Объединяем оба трансформера в единый ColumnTransformer
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, num_columns), ('cat', categorical_transformer, cat_columns)], n_jobs=-1)

# 4. Обучаем и применяем препроцессор к исходным данным
# (убедитесь, что в df[num_columns] нет пропусков NaN, например: df[num_columns] = df[num_columns].fillna(0))
encoded_features = preprocessor.fit_transform(df)

# 5. Преобразуем полученный массив в DataFrame с именами всех выходных признаков
transformed_df = pd.DataFrame(encoded_features, columns=preprocessor.get_feature_names_out())

# 6. Объединяем исходный DataFrame с преобразованными признаками
df = pd.concat([df, transformed_df], axis=1)

df.head(2)

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target,num__spl__monthly_charges_sp_0,num__spl__monthly_charges_sp_1,num__spl__monthly_charges_sp_2,num__spl__monthly_charges_sp_3,num__spl__monthly_charges_sp_4,num__spl__monthly_charges_sp_5,num__spl__total_charges_sp_0,num__spl__total_charges_sp_1,num__spl__total_charges_sp_2,num__spl__total_charges_sp_3,num__spl__total_charges_sp_4,num__spl__total_charges_sp_5,num__q__monthly_charges,num__q__total_charges,num__rb__monthly_charges,num__rb__total_charges,num__pol__1,num__pol__monthly_charges,num__pol__total_charges,num__pol__monthly_charges^2,num__pol__monthly_charges total_charges,num__pol__total_charges^2,num__pol__monthly_charges^3,num__pol__monthly_charges^2 total_charges,num__pol__monthly_charges total_charges^2,num__pol__total_charges^3,num__kbd__monthly_charges,num__kbd__total_charges,cat__type_One year,cat__type_Two year,cat__payment_method_Credit card (automatic),cat__payment_method_Electronic check,cat__payment_method_Mailed check,cat__internet_service_Fiber optic,cat__internet_service_None,cat__gender_Male
0,4,5575-GNVDE,2017-04-01,NaT,One year,No,Mailed check,56.95,1889.50,DSL,Yes,No,Yes,No,No,No,Male,0,No,No,No,0,0.000116,0.094742,0.554677,0.335807,0.014658,0.0,0.004242,0.228653,0.596333,0.169278,1.493678e-03,0.0,0.394114,0.581845,-0.246550,0.146087,1.0,56.95,1889.50,3243.3025,107607.0250,3.570210e+06,184706.077375,6.128220e+06,2.033235e+08,6.745912e+09,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,5,3668-QPYBK,2019-11-01,2020-01-01,Month-to-month,Yes,Mailed check,53.85,108.15,DSL,Yes,Yes,No,No,No,No,Male,0,No,No,No,1,0.000301,0.114432,0.572271,0.302499,0.010496,0.0,0.037668,0.445733,0.470623,0.045975,1.603155e-08,0.0,0.355221,0.120864,-0.303588,-0.379687,1.0,53.85,108.15,2899.8225,5823.8775,1.169642e+04,156155.441625,3.136158e+05,6.298524e+05,1.264968e+06,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


In [6]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.sklearn.log_model(preprocessor, "column_transformer") 

2026/11/19 06:09:07 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, log_loss
from catboost import CatBoostClassifier

# 1. Выделяем признаки и целевую переменную
# Берем все сгенерированные числовые признаки (num__ и cat__) + исходные числовые
feature_cols = [col for col in df.columns if col.startswith(('num__', 'cat__'))] + num_columns + ['senior_citizen']

X = df[feature_cols]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Обучаем модель
model = CatBoostClassifier(random_state=42, verbose=0)
model.fit(X_train, y_train)

# 3. Считаем метрики на тестовой выборке
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "roc_auc": roc_auc_score(y_test, y_proba),
    "f1": f1_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "logloss": log_loss(y_test, y_proba),
}
print("Метрики на тесте:", metrics)

# 4. Логируем и регистрируем модель в MLflow
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
experiment_id = experiment.experiment_id if experiment else mlflow.create_experiment(EXPERIMENT_NAME)

# Имя модели в реестре (можно использовать ту же константу REGISTRY_MODEL_NAME или имя из прошлых уроков)
MODEL_NAME = REGISTRY_MODEL_NAME

with mlflow.start_run(run_name="model_with_new_features", experiment_id=experiment_id) as run:
    current_run_id = run.info.run_id
    
    mlflow.log_metrics(metrics)
    mlflow.log_params(model.get_params())
    
    model_info = mlflow.catboost.log_model(
        cb_model=model,
        artifact_path="models",
        registered_model_name=MODEL_NAME,
        input_example=X_test.iloc[:5]
    )

print(f"RUN_ID: {current_run_id}")
print(f"REGISTERED_MODEL_NAME: {MODEL_NAME}")


Метрики на тесте: {'roc_auc': 0.8392492701955617, 'f1': 0.5619335347432024, 'precision': 0.6458333333333334, 'recall': 0.49732620320855614, 'logloss': 0.42310734341564765}


/home/mle-user/projects/mle-mlflow/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:333: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  input_schema = _infer_schema(input_ex)
/home/mle-user/projects/mle-mlflow/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:344: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Pytho

RUN_ID: 0eb29562f3b64fb0ae81c7e1d912e3ca
REGISTERED_MODEL_NAME: preprocessing_feature_engineering_lession_5_model


Created version '1' of model 'preprocessing_feature_engineering_lession_5_model'.
